In [1]:
!rm -rf /content/movie_genre_classification
!git clone https://github.com/oluwoleadetifa/movie_genre_classification.git
%cd /content/movie_genre_classification

import sys
sys.path.append("/content/movie_genre_classification")

Cloning into 'movie_genre_classification'...
remote: Enumerating objects: 297, done.
remote: Counting objects: 100% (112/112), done.
remote: Compressing objects: 100% (104/104), done.
remote: Total 297 (delta 57), reused 19 (delta 8), pack-reused 185 (from 1)
Receiving objects: 100% (297/297), 1.45 MiB | 4.66 MiB/s, done.
Resolving deltas: 100% (147/147), done.
/content/movie_genre_classification


In [2]:
!pip install transformers torch scikit-learn pandas numpy tqdm

In [6]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [8]:
import os
import shutil

# Path to Google Drive data
DRIVE_BASE = "/content/drive/MyDrive/Movie_Genre_Project"

# Source locations in Drive
PROCESSED_SRC = os.path.join(DRIVE_BASE, "dataset_processed", "movies_with_posters.csv")
SPLITS_SRC = os.path.join(DRIVE_BASE, "splits")

# Repo destination
PROJECT_BASE = "/content/movie_genre_classification/data"
PROCESSED_DST = os.path.join(PROJECT_BASE, "processed")
SPLITS_DST = os.path.join(PROJECT_BASE, "splits")

# Create destination folders
os.makedirs(PROCESSED_DST, exist_ok=True)
os.makedirs(SPLITS_DST, exist_ok=True)

# Copy processed dataset
shutil.copy(PROCESSED_SRC, os.path.join(PROCESSED_DST, "movies_with_posters.csv"))
print("Copied movies_with_posters.csv")

# Copy split files
for split_name in ["train.csv", "val.csv", "test.csv"]:
    src_file = os.path.join(SPLITS_SRC, split_name)
    dst_file = os.path.join(SPLITS_DST, split_name)

    if not os.path.exists(src_file):
        raise FileNotFoundError(f"Missing in Drive: {src_file}")

    shutil.copy(src_file, dst_file)
    print(f"Copied {split_name}")

Copied movies_with_posters.csv
Copied train.csv
Copied val.csv
Copied test.csv


In [9]:
import pandas as pd

train_df = pd.read_csv("/content/movie_genre_classification/data/splits/train.csv")
val_df = pd.read_csv("/content/movie_genre_classification/data/splits/val.csv")
test_df = pd.read_csv("/content/movie_genre_classification/data/splits/test.csv")

print(train_df.shape, val_df.shape, test_df.shape)
print(train_df.head())

(480, 6) (103, 6) (104, 6)
     movie_id                                        description    genre  \
0  tt20222202  Follows Eden, who goes to a coveted 'Heaven an...   action   
1  tt18411490  In 1999 in London, a teenage Zoya (Katrina Kai...   action   
2  tt15683734  An Indian-American teenager struggling with he...   horror   
3  tt13430858  In London, an award-winning film-maker documen...  romance   
4   tt2488666  Terror strikes when a team of paranormal inves...   horror   

                                          image_path  image_exists  label  
0  /Users/oluwoleadetifa/Library/CloudStorage/Goo...          True      0  
1  /Users/oluwoleadetifa/Library/CloudStorage/Goo...          True      0  
2  /Users/oluwoleadetifa/Library/CloudStorage/Goo...          True      2  
3  /Users/oluwoleadetifa/Library/CloudStorage/Goo...          True      3  
4  /Users/oluwoleadetifa/Library/CloudStorage/Goo...          True      2  


In [13]:
import torch
import numpy as np
import pandas as pd
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModel

# Config
TEXT_COLUMN = "description"
LABEL_COLUMN = "label"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

BASE_PATH = "/content/drive/MyDrive/Movie_Genre_Project/splits"

train_df = pd.read_csv(f"{BASE_PATH}/train.csv")
val_df = pd.read_csv(f"{BASE_PATH}/val.csv")
test_df = pd.read_csv(f"{BASE_PATH}/test.csv")

print("Train:", train_df.shape, "Val:", val_df.shape, "Test:", test_df.shape)

tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")
bert_model = AutoModel.from_pretrained("distilbert-base-uncased").to(device)
bert_model.eval()

Device: cuda
Train: (480, 6) Val: (103, 6) Test: (104, 6)


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


DistilBertModel(
  (embeddings): Embeddings(
    (word_embeddings): Embedding(30522, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (transformer): Transformer(
    (layer): ModuleList(
      (0-5): 6 x TransformerBlock(
        (attention): DistilBertSelfAttention(
          (q_lin): Linear(in_features=768, out_features=768, bias=True)
          (k_lin): Linear(in_features=768, out_features=768, bias=True)
          (v_lin): Linear(in_features=768, out_features=768, bias=True)
          (out_lin): Linear(in_features=768, out_features=768, bias=True)
          (dropout): Dropout(p=0.1, inplace=False)
        )
        (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
        (ffn): FFN(
          (dropout): Dropout(p=0.1, inplace=False)
          (lin1): Linear(in_features=768, out_features=3072, bias=True)
          (lin2): L

In [14]:
@torch.no_grad()
def encode_texts(texts, batch_size=16, max_length=256):
    embeddings = []

    for i in tqdm(range(0, len(texts), batch_size)):
        batch = list(texts[i:i + batch_size])

        encoded = tokenizer(
            batch,
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors="pt"
        )

        encoded = {k: v.to(device) for k, v in encoded.items()}
        outputs = bert_model(**encoded)

        hidden = outputs.last_hidden_state
        mask = encoded["attention_mask"].unsqueeze(-1)

        pooled = (hidden * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1)
        embeddings.append(pooled.cpu().numpy())

    return np.vstack(embeddings)

In [15]:
sample_embeddings = encode_texts(train_df[TEXT_COLUMN].tolist()[:8], batch_size=4)
print(sample_embeddings.shape)

100%|██████████| 2/2 [00:00<00:00,  3.14it/s]

(8, 768)


In [16]:
X_train_bert = encode_texts(train_df[TEXT_COLUMN].tolist(), batch_size=16)
X_val_bert = encode_texts(val_df[TEXT_COLUMN].tolist(), batch_size=16)
X_test_bert = encode_texts(test_df[TEXT_COLUMN].tolist(), batch_size=16)

y_train = train_df[LABEL_COLUMN].values
y_val = val_df[LABEL_COLUMN].values
y_test = test_df[LABEL_COLUMN].values

print(X_train_bert.shape, X_val_bert.shape, X_test_bert.shape)

100%|██████████| 7/7 [00:00<00:00, 19.61it/s]

(480, 768) (103, 768) (104, 768)


In [17]:
from sklearn.linear_model import LogisticRegression

clf = LogisticRegression(max_iter=1000)
clf.fit(X_train_bert, y_train)

LogisticRegression(max_iter=1000)

In [18]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

y_pred = clf.predict(X_test_bert)

print("BERT Test Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

BERT Test Accuracy: 0.8173076923076923
              precision    recall  f1-score   support

           0       0.84      0.86      0.85        36
           1       0.55      0.75      0.63        16
           2       0.94      0.89      0.92        38
           3       0.89      0.57      0.70        14

    accuracy                           0.82       104
   macro avg       0.80      0.77      0.77       104
weighted avg       0.84      0.82      0.82       104

Confusion Matrix:
[[31  4  1  0]
 [ 2 12  1  1]
 [ 3  1 34  0]
 [ 1  5  0  8]]


In [19]:
import json
import os

os.makedirs("/content/movie_genre_classification/outputs/metrics", exist_ok=True)

results_bert = {
    "test_accuracy": accuracy_score(y_test, y_pred),
    "classification_report": classification_report(y_test, y_pred, output_dict=True),
    "confusion_matrix": confusion_matrix(y_test, y_pred).tolist()
}

with open("/content/movie_genre_classification/outputs/metrics/bert_results.json", "w") as f:
    json.dump(results_bert, f, indent=2)

print("BERT results saved.")

BERT results saved.


In [22]:
import json
import os
from google.colab import files

input_path = "/content/movie_genre_classification/notebooks/05_bert_text_model.ipynb"
output_path = "/content/05_bert_text_model_clean.ipynb"

if not os.path.exists(input_path):
    raise FileNotFoundError(f"Notebook not found: {input_path}")

with open(input_path, "r", encoding="utf-8") as f:
    nb = json.load(f)

if "metadata" in nb:
    nb["metadata"].pop("widgets", None)

for cell in nb.get("cells", []):
    if "metadata" in cell:
        cell["metadata"].pop("widgets", None)
        cell["metadata"].pop("colab", None)
        cell["metadata"].pop("executionInfo", None)
        cell["metadata"].pop("id", None)

for cell in nb.get("cells", []):
    if cell.get("cell_type") == "code":
        cell["outputs"] = []
        cell["execution_count"] = None

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(nb, f, indent=2)

print(f"Clean notebook saved to: {output_path}")

files.download(output_path)

Clean notebook saved to: /content/05_bert_text_model_clean.ipynb


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>